In [1]:
DEFICIENCY_COLUMN_MAP = {
    'Calories': 'Calories',
    'Protein': 'Protein_g',
    'Fat': 'Fat_g',
    'Carbohydrates': 'Carbs_g',
    'Fiber': 'Fiber_g',
    'Vitamin A': 'Vitamin_A_mcg',
    'Vitamin C': 'Vitamin_C_mg',
    'Vitamin D': 'Vitamin_D_IU',
    'Vitamin B12': 'Vitamin_B12_mcg',
    'Iron': 'Iron_mg',
    'Calcium': 'Calcium_mg',
    'Zinc': 'Zinc_mg',
    'Folate': 'Folate_mcg',
    'Magnesium': 'Magnesium_mg'
    # Add more as needed based on your dataset's actual column names
}


In [2]:
import pandas as pd

def recommend_foods_for_deficiency(
    food_df,                 # pandas DataFrame of your nutrition dataset
    deficiencies,            # list/str of predicted deficiencies (e.g., ['Iron', 'Vitamin C'])
    top_n=5,                 # how many foods to show per deficiency
    exclude_allergens=None,  # list of allergens to exclude (optional)
    include_tags=None        # list of tags to require (optional)
):
    """
    For each deficiency, returns a DataFrame of top foods with nutrient value & health benefits.
    """
    result_dict = {}
    
    if isinstance(deficiencies, str):
        deficiencies = [deficiencies]

    for defic in deficiencies:
        nutr_col = DEFICIENCY_COLUMN_MAP.get(defic)
        if nutr_col is None or nutr_col not in food_df.columns:
            result_dict[defic] = pd.DataFrame([{'Message': f'No column found for {defic}'}])
            continue

        foods = food_df.loc[food_df[nutr_col] > 0].copy()

        if exclude_allergens:
            for allergen in exclude_allergens:
                foods = foods[~foods['Allergens'].str.contains(allergen, case=False, na=False)]

        if include_tags:
            mask = pd.Series(False, index=foods.index)
            for tag in include_tags:
                mask = mask | foods['Tags'].str.contains(tag, case=False, na=False)
            foods = foods[mask]
        
        foods = foods.sort_values(nutr_col, ascending=False)
        foods['Rank'] = range(1, len(foods)+1)
        display_cols = [
            'Rank', 'Food_Item', nutr_col, 'Calories', 'Category', 'Health_Benefits'
        ]
        # Add category, allergens, tags, etc., as desired
        display_cols = [col for col in display_cols if col in foods.columns]
        result = foods[display_cols].head(top_n)
        result.rename(columns={nutr_col: f"{defic} per 100g"}, inplace=True)
        result_dict[defic] = result.reset_index(drop=True)
    return result_dict


In [6]:
# Load your data (adjust file path as needed)
df = pd.read_csv(r"C:\My stuff\Coding\ML project\KiranveerSingh_Project\Dataset\cleaned_food_nutrition_dataset.csv").fillna(0)

# Example from ML model output:
predicted_deficiencies = ['Iron', 'Calcium', 'Vitamin B12']
# Optionally: exclude 'nuts' and require 'vegetarian'
recommendations = recommend_foods_for_deficiency(
    df, 
    predicted_deficiencies, 
    top_n=5, 
    exclude_allergens=['nuts'], 
    include_tags=['vegetarian']
)

# Display or export results for each deficiency
for defic, table in recommendations.items():
    print(f"\nTop foods for {defic} deficiency:")
    print(table.to_string(index=False))
    table.to_csv(r"C:\My stuff\Coding\ML project\KiranveerSingh_Project\Example\recommended_foods_{defic.lower().replace(' ', '_')}.csv", index=False)



Top foods for Iron deficiency:
 Rank            Food_Item  Iron per 100g  Calories Category      Health_Benefits
    1 Fish (Pomfret) (98g)            9.9     139.4  Seafood Supports Bone Health
    2    Groundnuts (108g)            9.8     369.5      Nut Supports Bone Health
    3         Guava (104g)            9.7     109.5    Fruit        Heart Healthy
    4        Cashews (87g)            9.7     435.4      Nut   Improves Digestion
    5     Brown Rice (96g)            9.7      71.4    Grain Supports Bone Health

Top foods for Calcium deficiency:
 Rank          Food_Item  Calcium per 100g  Calories  Category      Health_Benefits
    1      Spinach (87g)             299.5     255.2 Vegetable   Improves Digestion
    2       Apple (114g)             296.9     577.1     Fruit Supports Bone Health
    3 Cauliflower (101g)             291.5     540.4 Vegetable         Energy Boost
    4      Cabbage (98g)             290.5     348.0 Vegetable   Improves Digestion
    5      Cashews (9